<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


### Why this notebook exists
The chapter asks for an evaluation report, not a model dump. The notebook
should read like a small project write-up that a colleague could follow.

This cell continues the worked example for the current section.


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd

module_path = Path('code') / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

print(f'Loaded {module_path.resolve()}')


This cell continues the worked example for the current section.


In [ ]:
df = churn_report.load_customers()

print(df.head().to_string(index=False))
print()
print('churn counts:')
print(df[churn_report.TARGET_COLUMN].value_counts().sort_index())
print()
print('numeric summary:')
print(df[churn_report.NUMERIC_COLUMNS].describe().round(2).to_string())


This cell fits the model on the training split and checks the held-out rows.


In [ ]:
X_train, X_valid, X_test, y_train, y_valid, y_test = churn_report.split_customers(df)
models = churn_report.build_models()

rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    valid_scores = churn_report.evaluate_predictions(y_valid, valid_pred)
    test_scores = churn_report.evaluate_predictions(y_test, test_pred)

    rows.append({'model': name, 'split': 'validation', **valid_scores})
    rows.append({'model': name, 'split': 'test', **test_scores})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


This cell continues the worked example for the current section.


In [ ]:
validation_rows = comparison[comparison['split'] == 'validation'].copy()
best_name = validation_rows.sort_values(['recall', 'accuracy'], ascending=False).iloc[0]['model']
print('best validation model =', best_name)

best_model = fitted_models[best_name]
best_model.fit(pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid]))

mistakes = churn_report.misclassified_customers(best_model, df)
print()
print(mistakes.head().to_string(index=False))


### Report Summary
The model choice should be explained in plain language. A good summary
says what was tried, which model was chosen, what it still gets wrong, and
what data would help next.

### Where We Are Heading Next
The module ends here. The next step in a real project would be to package
the notebook, the data, and the short README into a small public repo.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
